In [8]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic     # Importamos la función geodesic para calcular distancias entre coordenadas geográficas

# Cargamos el dataset en formato parquet
df = pd.read_parquet("../data/pois_barcelona_procesados.parquet")

# Mostramos las primeras filas del dataset para comprobar que los datos se han cargado correctamente
df.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,...,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags,tags_str
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,...,OSM,True,False,True,True,3.21900,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,...,None,False,False,False,False,2.56796,False,60,"[culture, indoor]",culture|indoor
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,...,OSM,True,False,True,True,3.06500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,0.503766,...,None,False,False,False,False,2.68696,False,60,"[culture, indoor]",culture|indoor
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.750000,...,OSM,True,False,True,True,3.16500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor


In [9]:
# Verificamos que estám todas las columnas necesarias para realizar el modelado
df.columns

Index(['id', 'name', 'category', 'subcategory', 'latitude', 'longitude',
       'city', 'description', 'rating', 'match_confidence', 'opening_hours',
       'opening_hours_source', 'has_opening_hours', 'is_24_7',
       'is_likely_open', 'has_match_confidence', 'score', 'has_valid_source',
       'visit_duration', 'tags', 'tags_str'],
      dtype='object')

In [10]:
# Comprobamos los tipos de las columnas para verificar que son los correctos
df.dtypes

id                         int64
name                      object
category                category
subcategory             category
latitude                 float64
longitude                float64
city                      object
description               object
rating                   float64
match_confidence         float64
opening_hours             object
opening_hours_source      object
has_opening_hours           bool
is_24_7                     bool
is_likely_open              bool
has_match_confidence        bool
score                    float64
has_valid_source            bool
visit_duration             int64
tags                      object
tags_str                  object
dtype: object

In [11]:
# Creamos un nuevo DataFrame llamado df_base seleccionando solo las columnas relevantes  para simplificar el dataset y trabajar solo con lo necesario
df_base = df[[
    'id',
    'name',
    'category',
    'subcategory',
    'latitude',
    'longitude',
    'rating',
    'score',
    'visit_duration'
]].copy()   # hacemos una copia para no modificar el dataset original

# Eliminamos filas donde falten coordenadas (latitud o longitud) o porque luego vamos a calcular distancias geográficas
# Si hay valores NaN, la función geodesic falla
df_base = df_base.dropna(subset=['latitude', 'longitude'])

# Creamos una nueva columna llamada 'score_final' la cual será la métrica que usaremos para rankear los lugares
df_base['score_final'] = (
    df_base['rating'] * 0.6 +   # damos más peso al rating (60%)
    df_base['score'] * 0.4      # combinamos con el score interno (40%)
)

In [12]:
# Comprobamos los valores de la columna de categoria para hacer pruebas posteriores
df["category"].value_counts()

category
cultural              331
monument              100
tourist_attraction     84
fountain               75
park                   61
religious              44
historic               41
administrative         35
bridge                  3
unknown                 1
Name: count, dtype: int64

In [13]:
# Esta función sirve para obtener los mejores POIs según ciertos filtros
def recomendar(df, categoria=None, min_rating=0, top_n=10):
    
    # Hacemos una copia del DataFrame para no modificar el original
    data = df.copy()
    
    # Si el usuario especifica una categoría (por ejemplo "cultural") filtramos solo los POIs de esa categoría
    if categoria:
        data = data[data['category'] == categoria]
    
    # Filtramos los POIs que tengan un rating mínimo para evitar lugares con baja calidad
    data = data[data['rating'] >= min_rating]
    
    # Ordenamos los resultados por la columna 'score_final' de mayor a menor
    # Luego nos quedamos con los primeros N resultados (top_n)
    return data.sort_values(by='score_final', ascending=False).head(top_n)

# Ejecutamos la función (PRIMERA PRUEBA):
# - Solo POIs de categoría "cultural"
# - Con rating mínimo de 4
# - Mostramos los 5 mejores
recomendar(df_base, categoria='cultural', min_rating=4, top_n=5)

,id,name,category,subcategory,latitude,longitude,rating,score,visit_duration,score_final
225,6636,Palacio de la Música Catalana,cultural,theatre,41.3876,2.17522,4.9,3.709,120,4.4236
476,335226,Centre de Documentació Begoña Raventós -Biblio...,cultural,library,41.3850,2.15221,4.9,3.544,60,4.3576
471,335216,Bolsa de Barcelona,cultural,library,41.3899,2.16718,4.8,3.594,60,4.3176
292,7595,Museo del Chocolate (Barcelona),cultural,museum,41.3871,2.18174,4.8,3.552,120,4.3008
724,336266,Cases Rocamora,cultural,museum,41.3889,2.16940,4.8,3.504,120,4.2816


In [26]:
# Función para generar una ruta greedy
# El algoritmo parte de un punto inicial y en cada paso elige el POI más cercano
def generar_ruta(df, start_point, n=5):
    
    # Hacemos una copia del DataFrame para no modificar el original
    puntos = df.copy()
    
    # Eliminamos filas sin coordenadas válidas
    puntos = puntos.dropna(subset=['latitude', 'longitude'])
    
    # Lista donde guardaremos los POIs seleccionados
    ruta = []
    
    # Punto actual de partida
    actual = start_point
    
    # Repetimos hasta seleccionar n puntos o hasta quedarnos sin POIs
    for _ in range(min(n, len(puntos))):
        
        # Calculamos la distancia desde el punto actual hasta cada POI
        # Esta distancia es la que usa el algoritmo greedy para decidir el siguiente punto
        puntos['dist_actual_km'] = puntos.apply(
            lambda x: geodesic(actual, (x['latitude'], x['longitude'])).km,
            axis=1
        )
        
        # Calculamos la distancia desde el punto inicial del usuario hasta cada POI
        puntos['dist_inicio_km'] = puntos.apply(
            lambda x: geodesic(start_point, (x['latitude'], x['longitude'])).km,
            axis=1
        )
        
        # Seleccionamos el POI más cercano al punto actual
        siguiente = puntos.sort_values('dist_actual_km').iloc[0].copy()
        
        # Guardamos la distancia respecto al punto anterior de la ruta
        siguiente['dist_anterior_km'] = siguiente['dist_actual_km']
        
        # Añadimos el punto seleccionado a la ruta
        ruta.append(siguiente)
        
        # Actualizamos el punto actual con las coordenadas del POI elegido
        actual = (siguiente['latitude'], siguiente['longitude'])
        
        # Eliminamos el POI ya seleccionado para no repetirlo
        puntos = puntos.drop(siguiente.name)
    
    # Convertimos la lista de resultados en un DataFrame
    ruta_df = pd.DataFrame(ruta)
    
    # Si no hay resultados, devolvemos el DataFrame vacío
    if ruta_df.empty:
        return ruta_df
    
    # Calculamos distancia acumulada en km
    ruta_df['dist_acumulada_km'] = ruta_df['dist_anterior_km'].cumsum()
    
    # Convertimos todo a metros (numérico, mejor para ML)
    ruta_df['dist_desde_inicio_m'] = ruta_df['dist_inicio_km'] * 1000
    ruta_df['dist_desde_anterior_m'] = ruta_df['dist_anterior_km'] * 1000
    ruta_df['dist_acumulada_m'] = ruta_df['dist_acumulada_km'] * 1000
    
    # En la primera fila no hay punto anterior real
    ruta_df.loc[ruta_df.index[0], 'dist_desde_anterior_m'] = 0
    
    # Reordenamos columnas para que el resultado sea más claro
    columnas_finales = [
        'id',
        'name',
        'category',
        'subcategory',
        'rating',
        'score_final',
        'visit_duration',
        'latitude',
        'longitude',
        'dist_desde_anterior_m',
        'dist_desde_inicio_m',
        'dist_acumulada_m'
    ]
    
    # Nos quedamos solo con las columnas que existan en el DataFrame
    columnas_finales = [col for col in columnas_finales if col in ruta_df.columns]
    
    return ruta_df[columnas_finales]


# Punto inicial (ejemplo: centro de Barcelona)
start = (41.3851, 2.1734)

# Filtramos solo los POIs culturales con rating mínimo de 4
df_cultural = df_base[
    (df_base['category'] == 'cultural') &
    (df_base['rating'] >= 4.0)
].copy()

# Generamos una ruta de 5 POIs
ruta_cultural = generar_ruta(df_cultural, start, n=5)

# Reseteamos el índice para que empiece limpio
ruta_cultural = ruta_cultural.reset_index(drop=True)
ruta_cultural.index = ruta_cultural.index + 1

# Mostramos el resultado final
ruta_cultural

,id,name,category,subcategory,rating,score_final,visit_duration,latitude,longitude,dist_desde_anterior_m,dist_desde_inicio_m,dist_acumulada_m
1,335338,Teatre Tirso de Molina (Barcelona),cultural,library,4.6,4.116400,60,41.3852,2.17398,0.000000,49.768270,49.768270
2,5476,El mundo nace en cada beso,cultural,artwork,4.7,4.188800,15,41.3852,2.17482,70.260501,119.291914,120.028772
3,335281,Sant Josep Oriol,cultural,library,4.1,3.656000,60,41.3846,2.17451,71.503905,108.184104,191.532677
4,404,Real Círculo Artístico de Barcelona,cultural,arts_centre,4.2,3.765600,75,41.3845,2.17451,11.106135,114.283079,202.638812
5,335280,Biblioteca del Colegio de Arquitectos de Cataluña,cultural,library,4.8,4.073784,60,41.3844,2.17472,20.781890,135.034664,223.420702


In [25]:
df[df['name'].str.contains("Sagrada", case=False, na=False)]

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,...,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags,tags_str
246,7019,Templo Expiatorio de la Sagrada Familia,religious,cathedral,41.4035,2.17437,Barcelona,minor basilica,4.4,0.503766,...,None,False,False,False,False,2.74646,False,60,"[historic, religious]",historic|religious
262,7302,Escuelas de la Sagrada Familia,cultural,museum,41.4031,2.17425,Barcelona,school,4.3,0.800000,...,OSM,True,False,True,True,3.25000,True,120,"[culture, has_schedule, indoor, long_visit]",culture|has_schedule|indoor|long_visit
